In [4]:
import pandas as pd
import numpy as np
import joblib


class RecommendationEngine:

    def __init__(self):

        self.clustered_users = None
        self.courses = None
        self.transactions = None

    #########################################################
    # LOAD DATA
    #########################################################

    def load_data(

        self,

        clustered_users_path,

        courses_path,

        transactions_path

    ):

        self.clustered_users = pd.read_csv(
            clustered_users_path
        )

        self.courses = pd.read_csv(
            courses_path
        )

        self.transactions = pd.read_csv(
            transactions_path
        )

        print("Datasets Loaded Successfully")

        print("Clustered Users :", self.clustered_users.shape)

        print("Courses :", self.courses.shape)

        print("Transactions :", self.transactions.shape)

#########################################################
# GET USER PROFILE
#########################################################

    def get_user(self, user_id):

        user = self.clustered_users[

            self.clustered_users["UserID"] == user_id

        ]

        if user.empty:

            print("User Not Found")

            return None

        return user.iloc[0]

#########################################################
# GET USER CLUSTER
#########################################################

    def get_cluster(self, user_id):

        user = self.get_user(user_id)

        if user is None:

            return None

        return user["Cluster"]

#########################################################
# ENROLLED COURSES
#########################################################

    def enrolled_courses(self, user_id):

        enrolled = self.transactions[

            self.transactions["UserID"] == user_id

        ]["CourseID"].unique()

        return enrolled

#########################################################
# POPULAR COURSES
#########################################################

    def popular_courses(self, cluster):

        users = self.clustered_users[

            self.clustered_users["Cluster"] == cluster

        ]["UserID"]

        temp = self.transactions[

            self.transactions["UserID"].isin(users)

        ]

        popularity = temp.groupby("CourseID").size()

        popularity = popularity.reset_index()

        popularity.columns = [

            "CourseID",

            "Popularity"

        ]

        return popularity

#########################################################
# COURSE DETAILS
#########################################################

    def course_information(self):

        return self.courses[
            [
                "CourseID",
                "CourseName",
                "CourseCategory",
                "CourseLevel",
                "CourseRating"
            ]
        ]

#########################################################
    # GENERATE RECOMMENDATIONS
#########################################################
    
    def recommend_courses(
    
        self,
    
        user_id,
    
        category=None,
    
        level=None,
    
        top_n=10
    
    ):
    
        # Get User
        user = self.get_user(user_id)
    
        if user is None:
    
            return pd.DataFrame()
    
        # User Cluster
        cluster = user["Cluster"]
    
        # Already Enrolled
        enrolled = self.enrolled_courses(user_id)
    
        # Popularity
        popularity = self.popular_courses(cluster)
    
        # Course Details
        courses = self.course_information()
    
        # Merge
        recommendations = pd.merge(
    
            courses,
    
            popularity,
    
            on="CourseID",
    
            how="left"
    
        )
    
        # Fill Missing Popularity
        recommendations["Popularity"] = recommendations[
            "Popularity"
        ].fillna(0)
    
        #####################################################
        # Remove already enrolled courses
        #####################################################
    
        recommendations = recommendations[
            ~recommendations["CourseID"].isin(enrolled)
        ]
    
        #####################################################
        # Preferred Category
        #####################################################
    
        if category is None:
    
            category = user["PreferredCategory"]
    
        if category != "All":
    
            recommendations = recommendations[
    
                recommendations["CourseCategory"] == category
    
            ]
    
        #####################################################
        # Preferred Level
        #####################################################
    
        if level is None:
    
            level = user["PreferredLevel"]
    
        if level != "All":
    
            recommendations = recommendations[
    
                recommendations["CourseLevel"] == level
    
            ]
    
        #####################################################
        # Recommendation Score
        #####################################################
    
        recommendations["Score"] = (
    
            recommendations["Popularity"] * 0.6 +
    
            recommendations["CourseRating"] * 0.4
    
        )
    
        #####################################################
        # Sort
        #####################################################
    
        recommendations = recommendations.sort_values(
    
            by="Score",
    
            ascending=False
    
        )
    
        return recommendations.head(top_n)

#########################################################
    # DISPLAY RECOMMENDATIONS
#########################################################
    
    def show_recommendations(
    
        self,
    
        user_id,
    
        category=None,
    
        level=None,
    
        top_n=10
    
    ):
    
        recommendations = self.recommend_courses(
    
            user_id,
    
            category,
    
            level,
    
            top_n
    
        )
    
        print("\nRecommended Courses\n")
    
        print(
    
            recommendations[
    
                [
    
                    "CourseID",
    
                    "CourseName",
    
                    "CourseCategory",
    
                    "CourseLevel",
    
                    "CourseRating",
    
                    "Popularity",
    
                    "Score"
    
                ]
    
            ]
    
        )
    
        return recommendations

#########################################################
# SAVE RECOMMENDATIONS
#########################################################

    def save_recommendations(

        self,
    
        recommendations,
    
        filename="recommendations.csv"
    
    ):
    
        recommendations.to_csv(
    
            filename,
    
            index=False
    
        )
    
        print("Recommendations Saved")

############################################################
# MAIN
############################################################

engine = RecommendationEngine()

engine.load_data(

    "clustered_learners.csv",

    "dataset_EduPro/EduPro Online Platform.xlsx - Courses.csv",

    "dataset_EduPro/EduPro Online Platform.xlsx - Transactions.csv"

)

# Example User
user_id = "U00003"

recommendations = engine.show_recommendations(

    user_id=user_id,

    category=None,      # Uses learner's preferred category
    level=None,         # Uses learner's preferred level
    top_n=10

)

engine.save_recommendations(recommendations)

Datasets Loaded Successfully
Clustered Users : (3000, 16)
Courses : (60, 8)
Transactions : (10000, 7)

Recommended Courses

Empty DataFrame
Columns: [CourseID, CourseName, CourseCategory, CourseLevel, CourseRating, Popularity, Score]
Index: []
Recommendations Saved
